In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/xianggkl/VLU-Net.git
%cd VLU-Net

Cloning into 'VLU-Net'...
remote: Enumerating objects: 186, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 186 (delta 67), reused 180 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (186/186), 11.84 MiB | 15.08 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/VLU-Net


In [3]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers accelerate timm einops scikit-image pandas
!pip install -q open_clip_torch
!pip install -q lpips
!pip install scikit-video
!pip install pytorch-lightning
!pip install ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 66.7 MB/s eta 0:00:00


In [4]:
import os
import numpy as np
from PIL import Image
import torch

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [6]:
DATASET_ROOT = "/content/VLU-Net/datasets"
CACHE_ROOT = "/content/drive/MyDrive/VLU-Net-V2/datasets_cache"

os.makedirs(DATASET_ROOT, exist_ok=True)

!tar -xf "{CACHE_ROOT}/deblurring_datasets.tar" -C "{DATASET_ROOT}"
!tar -xf "{CACHE_ROOT}/dehazing_datasets.tar" -C "{DATASET_ROOT}"
!tar -xf "{CACHE_ROOT}/denoising_datasets.tar" -C "{DATASET_ROOT}"
!tar -xf "{CACHE_ROOT}/delowlight_datasets.tar" -C "{DATASET_ROOT}"
!tar -xf "{CACHE_ROOT}/deraining_datasets.tar" -C "{DATASET_ROOT}"

print("✓ All datasets extracted")

✓ All datasets extracted


In [20]:
!pip install -U bitsandbytes>=0.46.1 accelerate transformers
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 107.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [8]:
!jupyter nbconvert --to script /content/VLU-Net/CLIP_degradation_extractor.ipynb

[NbConvertApp] Converting notebook /content/VLU-Net/CLIP_degradation_extractor.ipynb to script
[NbConvertApp] Writing 7222 bytes to /content/VLU-Net/CLIP_degradation_extractor.py


In [21]:
%%writefile CLIP_degradation_extractor.py
import os
import torch
from torch import nn
import torch.optim as optim
import time
from tqdm import tqdm
import datasets
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig
import open_clip
import torch.nn.utils.rnn as rnn_utils
from argparse import ArgumentParser
from peft import PeftModel
import torchvision.transforms.functional as TF

class CLIP_Adapted_ImageEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip_model = clip_model

        self.adapter = nn.Sequential(
            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512)
        )

    def forward(self, image):
        return self.adapter(self.clip_model.encode_image(image))

class CLIP_Adapted_TextEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip_model = clip_model
        self.adapter = nn.Sequential(
            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512)
        )

    def forward(self, text):
        return self.adapter(self.clip_model.encode_text(text))

class DA_adapter(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        for param in clip_model.parameters():
            param.requires_grad = False
        self.ad_imageEncoder = CLIP_Adapted_ImageEncoder(clip_model=clip_model)
        self.ad_textEncoder = CLIP_Adapted_TextEncoder(clip_model=clip_model)

    def forward(self, image, text):
        image = self.ad_imageEncoder(image)
        text = self.ad_textEncoder(text)
        return image, text

parser = ArgumentParser(description='LDUN')
parser.add_argument('--about', type=str, default='5task')
parser.add_argument('--start_epoch', type=int, default=0, help='epoch number of start training')
parser.add_argument('--end_epoch', type=int, default=100, help='epoch number of end training')
parser.add_argument('--learning_rate', type=float, default=1e-5, help='learning rate')
parser.add_argument('--resume', type=bool, default=False, help='is resume')
parser.add_argument('--group_num', type=int, default=1, help='group number for training')
parser.add_argument('--gpu_list', type=str, default='1', help='gpu index')
parser.add_argument('--checkpoints_dir', type=str, default='/content/drive/MyDrive/VLU-Net-V2/Phase3/CLIP_degradation_extractor/Checkpoint', help='checkpoints dir')
parser.add_argument('--log_dir', type=str, default='log', help='log directory')
parser.add_argument('--ext', type=str, default='.png', help='training data directory')
parser.add_argument('--is_aug', type=bool,default=False, help='is aug')
parser.add_argument('--is_clip_tuning', type=bool,default=False, help='is finetuning clip')
parser.add_argument('--patch_size', type=int, default=128, help='patchsize of input.')

# Noise, Haze, Rain, Blurr, Lowlight
NHRBL = ["./datasets/denoising_datasets/15_train_paths.txt",
 "./datasets/denoising_datasets/25_train_paths.txt",
 "./datasets/denoising_datasets/50_train_paths.txt",
 "./datasets/dehazing_datasets/train_paths.txt",
 "./datasets/deraining_datasets/Rain100L/train_paths.txt",
 "./datasets/deblurring_datasets/GoPro/train_paths.txt",
 "./datasets/delowlight_datasets/LoL/train_paths.txt"]

args = parser.parse_args(args=[])
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu_list
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# !pip install --upgrade torchao

blip2_model_path = "/content/drive/MyDrive/VLU-Net-V2/Phase2/Blip2DegradationCaptionNew/blip2_finetuned_adapter"
print("Loading BLIP‑2 processor and model...")

model_name = "Salesforce/blip2-opt-2.7b"

processor = Blip2Processor.from_pretrained(model_name)
processor.tokenizer.pad_token = processor.tokenizer.eos_token


base_model = Blip2ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model_inf = PeftModel.from_pretrained(base_model, blip2_model_path)
model_inf.eval()
model_inf.to(device)
torch.set_grad_enabled(False)

model, preprocess, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
tokenizer = open_clip.get_tokenizer('ViT-B-32')

model = DA_adapter(model)
model.to(device)
model.eval()

@torch.inference_mode()
def generate_caption_batch(image_paths, model, processor, device):

    images = [
        Image.open(p).convert("RGB")
        for p in image_paths
    ]

    prompt = "Question: Describe the scene. Answer:"

    inputs = processor(
        images=images,
        text=[prompt] * len(images),
        return_tensors="pt",
        padding=True
    ).to(device)

    generated_ids = model.generate(
            **inputs,
            max_new_tokens=60,
            num_beams=1,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            length_penalty=0.8,
            early_stopping=True,
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.pad_token_id
        )

    captions = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    captions = [
        c.replace(prompt, "").strip()
        for c in captions
    ]

    return captions

BASE_DIR = "/content/VLU-Net/datasets"

def fix_path(p):
    return os.path.join(BASE_DIR, p.replace("./", "").strip())

all_image_paths = []

print("🔍 Building dataset paths...")

for file in tqdm(NHRBL, desc="📂 Dataset Files"):

    with open(file, "r") as f:
        lines = [l.strip() for l in f if l.strip()]

    for line in lines:

        if "," not in line:
            continue

        noisy_path, _ = line.split(",")

        noisy_path = fix_path(noisy_path)

        if os.path.exists(noisy_path):
            all_image_paths.append(noisy_path)

print("\n✅ TOTAL IMAGES:", len(all_image_paths))

BASE_SAVE = "/content/drive/MyDrive/VLU-Net-V2/Phase3/CLIP_degradation_extractor/Captions"

os.makedirs(BASE_SAVE, exist_ok=True)

CSV_PATH   = os.path.join(BASE_SAVE, "captions_dataset.csv")
CACHE_PATH = os.path.join(BASE_SAVE, "captions_cache.pt")
STATE_PATH = os.path.join(BASE_SAVE, "captions_state.pt")

SAVE_EVERY = 1000
BATCH_SIZE = 32

class DatasetManager:
    def __init__(self, csv_path, cache_path, state_path):

        self.csv_path = csv_path
        self.cache_path = cache_path
        self.state_path = state_path

        self.df = pd.DataFrame(columns=["image_path", "caption"])
        self.cache = {}
        self.state = {"processed": set()}

        self._load_all()

    def _load_all(self):

        # CSV
        if os.path.exists(self.csv_path):
            self.df = pd.read_csv(self.csv_path)
            print(f"[CSV] Loaded {len(self.df)} rows")

        # CACHE
        if os.path.exists(self.cache_path):
            self.cache = torch.load(self.cache_path)
            print(f"[CACHE] Loaded {len(self.cache)} items")

        # STATE
        if os.path.exists(self.state_path):
            self.state = torch.load(self.state_path)
            print(f"[STATE] Resumed {len(self.state['processed'])} items")

        # sync CSV → cache
        for _, row in self.df.iterrows():
            self.cache[row["image_path"]] = row["caption"]

    def add_batch(self, batch_data):
        """
        batch_data: list of (path, caption)
        """
        df_new = pd.DataFrame(batch_data, columns=["image_path", "caption"])
        self.df = pd.concat([self.df, df_new], ignore_index=True)

        for p, c in batch_data:
            self.cache[p] = c
            self.state["processed"].add(p)

    def exists(self, path):
        return path in self.cache or path in self.state["processed"]

    def save(self):
        self.df.to_csv(self.csv_path, index=False)
        torch.save(self.cache, self.cache_path)
        torch.save(self.state, self.state_path)

manager = DatasetManager(CSV_PATH, CACHE_PATH, STATE_PATH)

remaining_paths = [p for p in all_image_paths if not manager.exists(p)]

print("TOTAL:", len(all_image_paths))
print("REMAINING:", len(remaining_paths))

buffer = []

for i in tqdm(range(0, len(remaining_paths), BATCH_SIZE), desc="🖼️ Captioning"):

    batch_paths = remaining_paths[i:i+BATCH_SIZE]

    captions = generate_caption_batch(
        batch_paths,
        model_inf,
        processor,
        device
    )

    buffer.extend(list(zip(batch_paths, captions)))

    if len(buffer) >= SAVE_EVERY:
        manager.add_batch(buffer)
        buffer.clear()
        manager.save()

        print(f"💾 Saved {len(manager.df)} captions")

print("ALL IMAGES:", len(all_image_paths))
print("ALREADY DONE:", len(manager.state["processed"]))
print("TO PROCESS:", len([p for p in all_image_paths if not manager.exists(p)]))

if len(buffer) > 0:
    manager.add_batch(buffer)
    manager.save()

print("\n🎉 DONE")
print("Total captions:", len(manager.df))

clip_model, preprocess, _ = open_clip.create_model_and_transforms(
    'ViT-B-32',
    pretrained='laion2b_s34b_b79k'
)

model = DA_adapter(clip_model).to(device)
model.train()


for name, param in model.named_parameters():
    if "adapter" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.requires_grad for p in model.parameters())
print("Trainable params:", trainable)

class FastDataset(Dataset):
    def __init__(self, image_paths, captions_cache, preprocess):
        self.image_paths = image_paths
        self.captions_cache = captions_cache
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]

        image = Image.open(path).convert("RGB")
        image = self.preprocess(image)

        caption = self.captions_cache[path]

        return image, caption



captions_cache = torch.load(CACHE_PATH, map_location="cpu")
print("Loaded captions:", len(captions_cache))

train_image_paths = [
    p for p in all_image_paths
    if p in captions_cache
]

print("Train images:", len(train_image_paths))

dataset = FastDataset(train_image_paths,captions_cache,preprocess)

train_loader = DataLoader(dataset,batch_size=64,shuffle=True,num_workers=12,pin_memory=True,persistent_workers=True,prefetch_factor=4)
val_loader = DataLoader(dataset,batch_size=64,shuffle=True,num_workers=12,pin_memory=True,persistent_workers=True,prefetch_factor=4)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=args.learning_rate,
    betas=(0.9, 0.98),
    weight_decay=0.01
)

criterion = torch.nn.CrossEntropyLoss()
criterion = nn.CrossEntropyLoss()

temperature = 0.07

# =====================================================
# TEST ONE CAPTION + TOKENIZATION
# =====================================================

print("Caption cache loaded:", len(captions_cache))

sample_path = all_image_paths[0]

print("=" * 80)
print("SAMPLE IMAGE PATH:")
print(sample_path)

print("\nBLIP GENERATED CAPTION:")
print(captions_cache[sample_path])

tokens = tokenizer([captions_cache[sample_path]])

print("\nTOKEN TENSOR SHAPE:")
print(tokens.shape)

print("\nTOKEN IDS:")
print(tokens)

print("\nTOKEN IDS (LIST FORMAT):")
print(tokens[0].tolist())

print("=" * 80)


start_epoch = args.start_epoch

DRIVE_SAVE_DIR = args.checkpoints_dir

os.makedirs(
    DRIVE_SAVE_DIR,
    exist_ok=True
)

latest_ckpt_path = os.path.join(
    DRIVE_SAVE_DIR,
    "model_latest.pth"
)
best_ckpt_path = os.path.join(
    DRIVE_SAVE_DIR,
    "model_best.pth"
)

def load_checkpoint(ckpt_path):
    print(f"[INFO] Loading checkpoint: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])

    if "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    elif "optimizer" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer"])
    else:
        print("[WARNING] Optimizer state not found in checkpoint.")

    return checkpoint["epoch"] + 1

if os.path.exists(latest_ckpt_path):
    try:
        start_epoch = load_checkpoint(latest_ckpt_path)
        print(f"[INFO] Resume Epoch {start_epoch} from latest")
    except Exception as e:
        print(f"[ERROR] Failed to load latest checkpoint: {e}")
        if os.path.exists(best_ckpt_path):
            try:
                print("[INFO] Attempting to load best checkpoint instead...")
                start_epoch = load_checkpoint(best_ckpt_path)
                print(f"[INFO] Resume Epoch {start_epoch} from best")
            except Exception as e2:
                print(f"[ERROR] Failed to load best checkpoint as well: {e2}")
                print("[INFO] Training From Scratch")
        else:
            print("[INFO] Training From Scratch")
else:
    print("[INFO] Training From Scratch")

try:
    # Re-enable gradient tracking that was turned off during BLIP-2 loading
    torch.set_grad_enabled(True)

    # Get one batch from the training loader to define image_features and text_features
    images, captions = next(iter(train_loader))

    images = images.to(device, non_blocking=True)
    texts = tokenizer(list(captions)).to(device)

    # Ensure model is in training mode to track gradients
    model.train()

    # Perform a forward pass
    image_features, text_features = model(images, texts)

    # Calculate a dummy loss to define the 'loss' variable
    image_features_norm = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features_norm = text_features / text_features.norm(dim=-1, keepdim=True)
    logits = (image_features_norm @ text_features_norm.T) / temperature
    targets = torch.arange(len(images), device=device)
    loss = criterion(logits, targets)

    print("image requires_grad:", image_features.requires_grad)
    print("text requires_grad:", text_features.requires_grad)
    print("loss requires_grad:", loss.requires_grad)
except NameError as e:
    print(f"Error: {e}. Please ensure `train_loader`, `tokenizer`, `model`, `device`, `temperature`, and `criterion` are defined and accessible.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Re-enable gradient tracking
torch.set_grad_enabled(True)

train_losses = []
train_accs = []

val_losses = []
val_accs = []

best_val_acc = 0.0

for epoch in range(start_epoch, args.end_epoch):

    # TRAINING
    model.train()

    epoch_loss = 0.0
    epoch_correct = 0
    epoch_total = 0

    pbar = tqdm(train_loader, desc=f"Train Epoch {epoch}")

    for images, captions in pbar:

        images = images.to(device, non_blocking=True)
        texts = tokenizer(list(captions)).to(device)

        # Forward
        image_features, text_features = model(images, texts)

        image_features = image_features / image_features.norm(
            dim=-1, keepdim=True
        )
        text_features = text_features / text_features.norm(
            dim=-1, keepdim=True
        )

        logits = 100.0 * (image_features @ text_features.T)

        targets = torch.arange(len(images), device=device)

        loss = criterion(logits, targets)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        preds = logits.argmax(dim=1)
        batch_correct = (preds == targets).sum().item()

        epoch_correct += batch_correct
        epoch_total += len(images)

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{100 * batch_correct / len(images):.1f}%"
        })

    train_loss = epoch_loss / len(train_loader)
    train_acc = epoch_correct / epoch_total

    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # VALIDATION

    model.eval()

    val_epoch_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        pbar_val = tqdm(val_loader, desc=f"Val Epoch {epoch}")

        for images, captions in pbar_val:

            images = images.to(device, non_blocking=True)
            texts = tokenizer(list(captions)).to(device)

            image_features, text_features = model(images, texts)

            image_features = image_features / image_features.norm(
                dim=-1, keepdim=True
            )
            text_features = text_features / text_features.norm(
                dim=-1, keepdim=True
            )

            logits = 100.0 * (image_features @ text_features.T)

            targets = torch.arange(len(images), device=device)

            loss = criterion(logits, targets)

            val_epoch_loss += loss.item()

            preds = logits.argmax(dim=1)

            val_correct += (preds == targets).sum().item()
            val_total += len(images)

            pbar_val.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{100 * (preds == targets).sum().item() / len(images):.1f}%"
            })

    val_loss = val_epoch_loss / len(val_loader)
    val_acc = val_correct / val_total

    val_losses.append(val_loss)
    val_accs.append(val_acc)


    # PRINT METRICS
    print(
        f"\nEpoch {epoch} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc : {train_acc*100:.2f}% | "
        f"Val Loss  : {val_loss:.4f} | "
        f"Val Acc   : {val_acc*100:.2f}% | "
    )

    learnable_params = {
        name: param.detach().cpu()
        for name, param in model.named_parameters()
        if param.requires_grad
    }

    # SAVE LATEST CHECKPOINT
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "learnable_params": learnable_params,
        "train_losses": train_losses,
        "train_accs": train_accs,
        "val_losses": val_losses,
        "val_accs": val_accs,
        "best_val_acc": best_val_acc,
    }

    torch.save(checkpoint, latest_ckpt_path)

    # SAVE BEST CHECKPOINT
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        checkpoint["best_val_acc"] = best_val_acc

        torch.save(checkpoint, best_ckpt_path)

        print(
            f"[BEST] New best model saved "
            f"(Val Acc = {best_val_acc*100:.2f}%)"
        )

    print(
        f"[LATEST] Checkpoint saved "
        f"(Epoch {epoch})"
    )

Overwriting CLIP_degradation_extractor.py


In [22]:
!python CLIP_degradation_extractor.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading BLIP‑2 processor and model...
processor_config.json: 100% 68.0/68.0 [00:00<00:00, 308kB/s]
preprocessor_config.json: 100% 432/432 [00:00<00:00, 2.41MB/s]
config.json: 100% 1.03k/1.03k [00:00<00:00, 4.11MB/s]
tokenizer_config.json: 100% 882/882 [00:00<00:00, 5.46MB/s]
vocab.json: 100% 798k/798k [00:00<00:00

In [28]:
!mkdir -p /content/VLU-Net/Checkpoint
!cp /content/drive/MyDrive/VLU-Net-V2/Phase3/CLIP_degradation_extractor/Checkpoint/model_best.pth \
    /content/VLU-Net/Checkpoint/

In [63]:
%%writefile train.py
# %%
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from utils.dataset_utils_clip import PromptTrainDataset, TestDataset_forIR
from net.Final import VLUNet
from utils.schedulers import LinearWarmupCosineAnnealingLR
# import wandb
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from utils.val_utils import AverageMeter, compute_psnr_ssim
from utils.image_io import save_image_tensor
import os
import argparse
import open_clip
from net.clip import *
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"

from pytorch_lightning.callbacks import TQDMProgressBar
from tqdm.auto import tqdm
from pytorch_lightning.loggers import TensorBoardLogger
import glob

torch.set_float32_matmul_precision("high")

# %%
parser = argparse.ArgumentParser()
# Input Parameters
parser.add_argument('--cuda', type=int, default=0)
parser.add_argument('--name', type=str, default="Delowlight")

parser.add_argument('--epochs', type=int, default=200, help='maximum number of epochs to train the total model.')
parser.add_argument('--batch_size', type=int,default=4,help="Batch size to use per GPU")
parser.add_argument('--lr', type=float, default=2e-4, help='learning rate of encoder.')

parser.add_argument('--de_type', nargs='+', default=['delowlight'],
                    help='which type of degradations is training and testing for.')
# parser.add_argument('--de_type', nargs='+', default=['denoise_15', 'denoise_25', 'denoise_50', 'derain', 'dehaze'],
#                     help='which type of degradations is training and testing for.')
# parser.add_argument('--de_type', nargs='+', default=['denoise_15', 'denoise_25', 'denoise_50'],
#                     help='which type of degradations is training and testing for.')
parser.add_argument('--de_dim', type=int, default=7)

parser.add_argument('--patch_size', type=int, default=128, help='patchsize of input.')
parser.add_argument('--num_workers', type=int, default=16, help='number of workers.')
parser.add_argument("--num_gpus",type=int,default=1,help = "Number of GPUs to use for training")

# path
parser.add_argument('--denoise15_dir', type=str, default='./datasets/denoising_datasets/15_train_paths.txt',
                    help='paths for 15 noise and clean, where clean images of denoising saves.')
parser.add_argument('--denoise25_dir', type=str, default='./datasets/denoising_datasets/25_train_paths.txt',
                    help='paths for 25 noise and clean, where clean images of denoising saves.')
parser.add_argument('--denoise50_dir', type=str, default='./datasets/denoising_datasets/50_train_paths.txt',
                    help='paths for 50 noise and clean, where clean images of denoising saves.')
parser.add_argument('--derain_dir', type=str, default='./datasets/deraining_datasets/train_paths.txt',
                    help='where training images of deraining saves.')
parser.add_argument('--dehaze_dir', type=str, default='./datasets/dehazing_datasets/train_paths.txt',
                    help='hazeimages and clean, where training images of dehazing saves.')
parser.add_argument('--deblur_dir', type=str, default='./datasets/deblurring_datasets/GoPro/train_paths.txt',
                    help='blurimages and clean, where training images of dehazing saves.')
parser.add_argument('--delowlight_dir', type=str, default='./datasets/delowlight_datasets/LoL/train_paths.txt',
                    help='lowlightimages and clean, where training images of dehazing saves.')

parser.add_argument("--is_addRainSets",type=bool,default=False, help = "is added datasets for rain is added")
parser.add_argument('--is_clip_tuning', type=bool,default=True, help='is finetuning clip')
parser.add_argument('--is_clip', type=bool,default=True, help='is clip')

parser.add_argument('--output_path', type=str, default="output/", help='output save path')
parser.add_argument('--ckpt_path', type=str, default="ckpt/", help='checkpoint save path')
options = parser.parse_args(args=[])

options.output_path = os.path.join(options.output_path, options.name)
options.ckpt_path = os.path.join(options.ckpt_path, options.name)
os.makedirs(options.output_path, exist_ok=True)
os.makedirs(options.ckpt_path, exist_ok=True)

if options.is_addRainSets == False:
    options.derain_dir = "./datasets/deraining_datasets/Rain100L/train_paths.txt"

# %%
class DAdunModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.net = VLUNet(options.de_dim)

        if options.is_clip:
            clip, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
            if options.is_clip_tuning:
                self.model_degradation = DA_adapter(clip)
                self.model_degradation.set_frozen()
                pth_file = "./Checkpoint/model_best.pth"
                checkpoint = torch.load(pth_file, map_location=torch.device('cpu'))
                self.model_degradation.load_state_dict(checkpoint['learnable_params'], strict=False)
            else:
                for param in clip.parameters():
                    param.requires_grad = False
                self.model_degradation = clip

        self.loss_fn  = nn.L1Loss()
        self.validation_step_outputs = []

    def forward(self,x, clip_input):
        degradation_features = ""
        if options.is_clip:
            if options.is_clip_tuning:
                degradation_features = self.model_degradation.get_image_features(clip_input)
            else:
                degradation_features = self.model_degradation.encode_image(clip_input)

        return self.net(x, degradation_features)

    def training_step(self, batch, batch_idx):
        ([clean_name, de_id], degrad_patch, clean_patch, clip_input) = batch

        degradation_features = ""
        if options.is_clip:
            if options.is_clip_tuning:
                degradation_features = self.model_degradation.get_image_features(clip_input)
            else:
                degradation_features = self.model_degradation.encode_image(clip_input)

        restored = self.net(degrad_patch, degradation_features)

        loss = self.loss_fn(restored,clean_patch)
        # Logging to TensorBoard (if installed) by default
        # self.log("train_loss", loss)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        return loss

    def lr_scheduler_step(self,scheduler,metric):
        scheduler.step(self.current_epoch)
        lr = scheduler.get_last_lr()

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=2e-4)
        scheduler = LinearWarmupCosineAnnealingLR(optimizer=optimizer,warmup_epochs=15,max_epochs=options.epochs)
        return [optimizer],[scheduler]

    def validation_step(self, batch, batch_idx, dataloader_idx=0):
        (input, target, clip_input, type, name, sets_name) = batch

        degradation_features = ""
        if options.is_clip:
            if options.is_clip_tuning:
                degradation_features = self.model_degradation.get_image_features(clip_input)
            else:
                degradation_features = self.model_degradation.encode_image(clip_input)

        restored = self.net(input, degradation_features)
        val_loss = self.loss_fn(restored, target)

        temp_psnr, temp_ssim, N = compute_psnr_ssim(restored, target)
        output_path = os.path.join(options.output_path, sets_name[0])
        os.makedirs(output_path, exist_ok=True)
        output_path = os.path.join(output_path, type[0])
        os.makedirs(output_path, exist_ok=True)
        save_image_tensor(restored, os.path.join(output_path, name[0]))

        out_dict ={
            'dataloader_idx': dataloader_idx,
            'val_loss': val_loss.item(),
            'temp_psnr': temp_psnr,
            'temp_ssim': temp_ssim
        }

        self.validation_step_outputs.append(out_dict)

        return out_dict

    def on_validation_epoch_end(self):
        print("\n")

        if self.trainer.sanity_checking:
            print("Sanity check complete. Skipping validation metrics computation.")
            self.validation_step_outputs.clear()
            return

        metrics = {}
        total_psnr_sum = 0
        total_val_loss_sum = 0
        total_count = 0

        for output in self.validation_step_outputs:
            if not output or 'dataloader_idx' not in output:
                continue
            dataloader_idx = output['dataloader_idx']
            psnr = output['temp_psnr']
            ssim = output['temp_ssim']
            val_loss = output['val_loss']

            if dataloader_idx not in metrics:
                metrics[dataloader_idx] = {'psnr_sum': 0, 'ssim_sum': 0, 'val_loss_sum': 0, 'count': 0}

            metrics[dataloader_idx]['psnr_sum'] += psnr
            metrics[dataloader_idx]['ssim_sum'] += ssim
            metrics[dataloader_idx]['val_loss_sum'] += val_loss
            metrics[dataloader_idx]['count'] += 1

            total_psnr_sum += psnr
            total_val_loss_sum += val_loss
            total_count += 1

            # 计算平均值并记录
        for idx, metric in metrics.items():
            avg_psnr = metric['psnr_sum'] / metric['count']
            avg_ssim = metric['ssim_sum'] / metric['count']
            avg_val_loss = metric['val_loss_sum'] / metric['count']
            self.log(f'avg_val_psnr_dataloader_{idx}', avg_psnr, sync_dist=True)
            self.log(f'avg_val_ssim_dataloader_{idx}', avg_ssim, sync_dist=True)
            self.log(f'avg_val_loss_dataloader_{idx}', avg_val_loss, sync_dist=True)
            print(f"Dataloader {idx}, Loss: {avg_val_loss:.4f}, PSNR: {avg_psnr:.2f}, SSIM: {avg_ssim:.4f}")


        if total_count > 0:
          global_avg_psnr = total_psnr_sum / total_count
          global_avg_val_loss = total_val_loss_sum / total_count
          self.log('val_psnr',global_avg_psnr, sync_dist=True)
          self.log('val_loss', global_avg_val_loss, sync_dist=True)
        self.validation_step_outputs.clear()

class CustomProgressBar(TQDMProgressBar):
    def init_validation_tqdm(self):
        bar = super().init_validation_tqdm()
        bar.leave = False
        return bar

# %%
test_loaders = []
# test_path = './datasets/denoising_datasets/CBSD68/15_test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Denoise15", "CBSD68")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# test_path = './datasets/denoising_datasets/CBSD68/25_test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Denoise25", "CBSD68")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# test_path = './datasets/denoising_datasets/CBSD68/50_test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Denoise50", "CBSD68")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# test_path = './datasets/dehazing_datasets/test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Dehazing", "SOTS_outdoors")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# test_path = './datasets/deraining_datasets/Rain100L/test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Deraining", "Rain100L")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# test_path = './datasets/deblurring_datasets/GoPro/test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Deblurring", "GoPro")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

test_path = './datasets/delowlight_datasets/LoL/test_paths.txt'
test_dataset = TestDataset_forIR(test_path, "Delowlight", "LoL")
test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# %%
def main():
    tb_logger = TensorBoardLogger(
        save_dir="/content/drive/MyDrive/VLU-Net-V2/Phase4/Logs",
        name=options.name,
        version="log_1"
    )

    drive_target_dir = f"/content/drive/MyDrive/VLU-Net-V2/Phase4/Checkpoint/Delowlight"
    os.makedirs(drive_target_dir, exist_ok=True)

    trainset = PromptTrainDataset(options)
    checkpoint_callback = ModelCheckpoint(dirpath = options.ckpt_path,every_n_epochs = 5,save_top_k=-1)
    trainloader = DataLoader(trainset, batch_size=options.batch_size, pin_memory=True, shuffle=True,
                             drop_last=True, num_workers=options.num_workers)
    model = DAdunModel()

    latest_callback = ModelCheckpoint(
        dirpath=drive_target_dir,
        filename="latest",
        every_n_epochs=1,
        save_top_k=1,
        monitor=None,
        save_on_train_epoch_end=True,
        enable_version_counter=False
    )

    best_callback = ModelCheckpoint(
        dirpath=drive_target_dir,
        filename="best-{epoch:03d}-{avg_val_psnr_dataloader_0:.3f}",
        monitor="avg_val_psnr_dataloader_0",
        mode="max",
        save_top_k=1
    )

    resume_path = os.path.join(drive_target_dir, "latest.ckpt")
    if os.path.exists(resume_path):
        print(f"[RESUME INFO] Found uncompleted runtime session checkpoint at {resume_path}. Resuming pipeline...")
    else:
        print("[RESUME INFO] Initialization fallback: no current checkpoint detected. Starting standard initialization.")
        resume_path = None

    trainer = pl.Trainer(
        max_epochs=options.epochs,
        accelerator="gpu",
        devices=options.num_gpus,
        logger=tb_logger,
        precision="16-mixed",
        callbacks=[latest_callback, best_callback, CustomProgressBar()],
        check_val_every_n_epoch=5
    )

    trainer.fit(model=model, train_dataloaders=trainloader, val_dataloaders=test_loaders, ckpt_path=resume_path)

    print("Best score:", best_callback.best_model_score)
    print("Best path:", best_callback.best_model_path)

if __name__ == '__main__':
    main()


Overwriting train.py


In [64]:
!python train.py --name Delowlight --de_dim 7 --de_type ['delowlight']

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
['delowlight']
Total lowlight Ids : 485
485
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[RESUME INFO] Initialization fallback: no current checkpoint detected. Starting standard initialization.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU availabl

In [65]:
!mkdir -p /content/drive/MyDrive/VLU-Net-V2/Phase4/Checkpoint/BlipVLUNet
!cp -r /content/drive/MyDrive/VLU-Net-V2/Phase4/Checkpoint/Delowlight/best-epoch*.ckpt \
    /content/drive/MyDrive/VLU-Net-V2/Phase4/Checkpoint/BlipVLUNet/single_lowlight_blip_vlunet.ckpt

In [72]:
%%writefile test.py
# %%
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import sys

from utils.dataset_utils_clip import PromptTrainDataset, TestDataset_forIR
from net.Final import VLUNet
from utils.schedulers import LinearWarmupCosineAnnealingLR
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from utils.val_utils import AverageMeter, compute_psnr_ssim
from utils.image_io import save_image_tensor
import os
import argparse
import open_clip
from net.clip import *
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# %%
parser = argparse.ArgumentParser()
# Input Parameters
parser.add_argument('--cuda', type=int, default=0)
parser.add_argument('--name', type=str, default="final_results")
parser.add_argument('--task', type=str, default="NHR")

parser.add_argument('--epochs', type=int, default=200, help='maximum number of epochs to train the total model.')
parser.add_argument('--batch_size', type=int,default=4,help="Batch size to use per GPU")
parser.add_argument('--lr', type=float, default=2e-4, help='learning rate of encoder.')

parser.add_argument('--de_type', nargs='+', default=['delowlight'],
                    help='which type of degradations is training and testing for.')
# parser.add_argument('--de_type', nargs='+', default=['denoise_15', 'denoise_25', 'denoise_50', 'derain', 'dehaze', 'delowlight', 'deblur'],
#                     help='which type of degradations is training and testing for.')
parser.add_argument('--de_dim', type=int, default=7)

parser.add_argument('--patch_size', type=int, default=128, help='patchsize of input.')
parser.add_argument('--num_workers', type=int, default=16, help='number of workers.')
parser.add_argument("--num_gpus",type=int,default=0,help = "Number of GPUs to use for training")

# path
parser.add_argument('--denoise15_dir', type=str, default='./datasets/denoising_datasets/15_train_paths.txt',
                    help='paths for 15 noise and clean, where clean images of denoising saves.')
parser.add_argument('--denoise25_dir', type=str, default='./datasets/denoising_datasets/25_train_paths.txt',
                    help='paths for 25 noise and clean, where clean images of denoising saves.')
parser.add_argument('--denoise50_dir', type=str, default='./datasets/denoising_datasets/50_train_paths.txt',
                    help='paths for 50 noise and clean, where clean images of denoising saves.')
parser.add_argument('--derain_dir', type=str, default='./datasets/deraining_datasets/train_paths.txt',
                    help='where training images of deraining saves.')
parser.add_argument('--dehaze_dir', type=str, default='./datasets/dehazing_datasets/train_paths.txt',
                    help='hazeimages and clean, where training images of dehazing saves.')
parser.add_argument('--deblur_dir', type=str, default='./datasets/deblurring_datasets/GoPro/train_paths.txt',
                    help='blurimages and clean, where training images of dehazing saves.')
parser.add_argument('--delowlight_dir', type=str, default='./datasets/delowlight_datasets/LoL/train_paths.txt',
                    help='lowlightimages and clean, where training images of dehazing saves.')

parser.add_argument("--is_addRainSets",type=bool,default=False, help = "is added datasets for rain is added")
parser.add_argument('--is_clip_tuning', type=bool,default=True, help='is finetuning clip')
parser.add_argument('--is_clip', type=bool,default=True, help='is clip')

parser.add_argument('--output_path', type=str, default="output/", help='output save path')
parser.add_argument('--ckpt_path', type=str, default="ckpt/", help='checkpoint save path')
parser.add_argument('--pretrained_ckpt_path', type=str, default="./pretrained_ckpt/3task_vlunet.ckpt", help='checkpoint save path')
if hasattr(sys, 'ps1') or 'ipykernel' in sys.modules:
    options = parser.parse_args(args=[])
else:
    options = parser.parse_args()

options.output_path = os.path.join(options.output_path, options.name)
options.ckpt_path = os.path.join(options.ckpt_path, options.name)
os.makedirs(options.output_path, exist_ok=True)
os.makedirs(options.ckpt_path, exist_ok=True)

options.output_path = os.path.join(options.output_path, options.task)
os.makedirs(options.output_path, exist_ok=True)
if options.is_addRainSets == False:
    options.derain_dir = "./datasets/deraining_datasets/Rain100L/train_paths.txt"

# %%
class DAdunModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.net = VLUNet(options.de_dim)

        if options.is_clip:
            clip, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
            if options.is_clip_tuning:
                self.model_degradation = DA_adapter(clip)
                self.model_degradation.set_frozen()
                pth_file = "./Checkpoint/model_best.pth"
                checkpoint = torch.load(pth_file, map_location=torch.device('cpu'))
                self.model_degradation.load_state_dict(checkpoint['learnable_params'], strict=False)
            else:
                for param in clip.parameters():
                    param.requires_grad = False
                self.model_degradation = clip

        self.loss_fn  = nn.L1Loss()
        self.test_step_outputs = []

    def forward(self,x, clip_input):
        degradation_features = ""
        if options.is_clip:
            if options.is_clip_tuning:
                degradation_features = self.model_degradation.get_image_features(clip_input)
            else:
                degradation_features = self.model_degradation.encode_image(clip_input)

        return self.net(x, degradation_features)

    def training_step(self, batch, batch_idx):
        ([clean_name, de_id], degrad_patch, clean_patch, clip_input) = batch

        degradation_features = ""
        if options.is_clip:
            if options.is_clip_tuning:
                degradation_features = self.model_degradation.get_image_features(clip_input)
            else:
                degradation_features = self.model_degradation.encode_image(clip_input)

        restored = self.net(degrad_patch, degradation_features)

        loss = self.loss_fn(restored,clean_patch)
        # Logging to TensorBoard (if installed) by default
        self.log("train_loss", loss)
        return loss

    def lr_scheduler_step(self,scheduler,metric):
        scheduler.step(self.current_epoch)
        lr = scheduler.get_last_lr()

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=2e-4)
        scheduler = LinearWarmupCosineAnnealingLR(optimizer=optimizer,warmup_epochs=15,max_epochs=options.epochs)

        return [optimizer],[scheduler]

    def test_step(self, batch, batch_idx, dataloader_idx=0):
        (input, target, clip_input, type, name, sets_name) = batch

        degradation_features = ""
        if options.is_clip:
            if options.is_clip_tuning:
                degradation_features = self.model_degradation.get_image_features(clip_input)
            else:
                degradation_features = self.model_degradation.encode_image(clip_input)

        restored = self.net(input, degradation_features)

        temp_psnr, temp_ssim, N = compute_psnr_ssim(restored, target)
        output_path = os.path.join(options.output_path, sets_name[0])
        os.makedirs(output_path, exist_ok=True)
        output_path = os.path.join(output_path, type[0])
        os.makedirs(output_path, exist_ok=True)
        save_image_tensor(restored, os.path.join(output_path, name[0]))

        self.test_step_outputs.append({
            'dataloader_idx': dataloader_idx,
            'temp_psnr': temp_psnr,
            'temp_ssim': temp_ssim
        })

    def on_test_epoch_end(self):
        print("\n")
        metrics = {}
        for output in self.test_step_outputs:
            dataloader_idx = output['dataloader_idx']
            psnr = output['temp_psnr']
            ssim = output['temp_ssim']

            if dataloader_idx not in metrics:
                metrics[dataloader_idx] = {'psnr_sum': 0, 'ssim_sum': 0, 'count': 0}

            metrics[dataloader_idx]['psnr_sum'] += psnr
            metrics[dataloader_idx]['ssim_sum'] += ssim
            metrics[dataloader_idx]['count'] += 1

        # 计算平均值并记录
        for idx, metric in metrics.items():
            avg_psnr = metric['psnr_sum'] / metric['count']
            avg_ssim = metric['ssim_sum'] / metric['count']
            print(f"Dataloader {idx}, PSNR: {avg_psnr:.2f}, SSIM: {avg_ssim:.4f}")
        self.test_step_outputs.clear()

# %%
test_loaders = []
# test_path = './datasets/denoising_datasets/CBSD68/15_test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Denoise15", "CBSD68")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# test_path = './datasets/denoising_datasets/CBSD68/25_test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Denoise25", "CBSD68")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# test_path = './datasets/denoising_datasets/CBSD68/50_test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Denoise50", "CBSD68")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# test_path = './datasets/dehazing_datasets/test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Dehazing", "SOTS_outdoors")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# test_path = './datasets/deraining_datasets/Rain100L/test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Deraining", "Rain100L")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# test_path = './datasets/deblurring_datasets/GoPro/test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Deblurring", "GoPro")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

test_path = './datasets/delowlight_datasets/LoL/test_paths.txt'
test_dataset = TestDataset_forIR(test_path, "Delowlight", "LoL")
test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))


# test_path = './datasets/denoising_datasets/CBSD68/rand_test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Denoise_rand", "CBSD68")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# test_path = './datasets/denoising_datasets/Urban100_HR/rand_test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Denoise_rand", "Urban100_HR")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))


# test_path = './datasets/denoising_datasets/Urban100_HR/15_test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Denoise15", "Urban100_HR")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=3))

# test_path = './datasets/denoising_datasets/Urban100_HR/25_test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Denoise25", "Urban100_HR")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=3))

# test_path = './datasets/denoising_datasets/Urban100_HR/50_test_paths.txt'
# test_dataset = TestDataset_forIR(test_path, "Denoise50", "Urban100_HR")
# test_loaders.append(DataLoader(test_dataset, batch_size=1, num_workers=1))

# %%
def main():
    logger = WandbLogger(
        project="DAdun",
        name=options.name,
        offline=False,
        config=vars(options),
        log_model="checkpoint"
    )
    os.environ["WANDB_DISABLED"] = "true"

    checkpoint_path = options.pretrained_ckpt_path
    model = DAdunModel.load_from_checkpoint(checkpoint_path)

    trainer = pl.Trainer(accelerator="gpu", devices=1)

    trainer.test(model, dataloaders=test_loaders)

if __name__ == '__main__':
    main()

Overwriting test.py


In [73]:
!python test.py --name Delowlight --task blip_L --de_dim 7 --pretrained_ckpt_path "/content/drive/MyDrive/VLU-Net-V2/Phase4/Checkpoint/BlipVLUNet/single_lowlight_blip_vlunet.ckpt"

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for p